# Retail Sales Quality and Credits

Completed audit and SQL analysis of historical UCI Online Retail invoice lines. Run the preparation script first, then run this notebook from top to bottom.

Source: Chen (2015), [UCI Online Retail](https://archive.ics.uci.edu/dataset/352/online%2Bretail), CC BY 4.0.


## Load the prepared data

`source_row` identifies the original worksheet row. Invoice, stock and customer identifiers are loaded as strings. Raw records remain unchanged.


In [1]:
from pathlib import Path
import pandas as pd

# Locate the prepared project from its root or a notebook subfolder.
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in candidates if (p / "data/raw/online_retail.csv").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run scripts/prepare_data.py, then open this notebook inside the project.")


In [2]:
df = pd.read_csv(
    PROJECT_ROOT / "data/raw/online_retail.csv",
    dtype={"InvoiceNo": "string", "StockCode": "string", "CustomerID": "string"},
    parse_dates=["InvoiceDate"],
)


## Inspect source records and schema

Invoice lines are not equivalent to distinct orders or customers.


In [3]:
# inspect the first 10 rows
df.head(10)

,source_row,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,2,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,3,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,4,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,5,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,6,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
5,7,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom
6,8,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850,United Kingdom
7,9,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850,United Kingdom
8,10,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850,United Kingdom
9,11,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047,United Kingdom


In [4]:
# inspect the shape
df.shape

(541909, 9)

In [5]:
# inspect the column names and data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   source_row   541909 non-null  int64         
 1   InvoiceNo    541909 non-null  string        
 2   StockCode    541909 non-null  string        
 3   Description  540455 non-null  object        
 4   Quantity     541909 non-null  int64         
 5   InvoiceDate  541909 non-null  datetime64[ns]
 6   UnitPrice    541909 non-null  float64       
 7   CustomerID   406829 non-null  string        
 8   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(2), string(3)
memory usage: 37.2+ MB


### 3. Record your interpretation

- 135,080 rows lack a CustomerID — approximately 24.9%. We should retain these for sales analysis, but they limit customer analysis.
- 1,454 rows lack a Description. We need to inspect them before deciding how to treat them.

## Audit exceptional records and define reporting scope

The following checks distinguish credits, zero-price records, bad-debt adjustments and possible repeats before calculating merchandise measures.


In [6]:
checks = pd.Series({
    "Earliest date": df["InvoiceDate"].min(),
    "Latest date": df["InvoiceDate"].max(),
    "Distinct invoices": df["InvoiceNo"].nunique(),
    "Known customers": df["CustomerID"].nunique(),
    "Negative quantities": df["Quantity"].lt(0).sum(),
    "Zero quantities": df["Quantity"].eq(0).sum(),
    "Negative prices": df["UnitPrice"].lt(0).sum(),
    "Zero prices": df["UnitPrice"].eq(0).sum(),
    "Invoice lines beginning C": (
        df["InvoiceNo"].str.upper().str.startswith("C").sum()
    ),
    "Repeated rows after first occurrence": (
        df.drop(columns="source_row").duplicated().sum()
    ),
})

checks

Earliest date                           2010-12-01 08:26:00
Latest date                             2011-12-09 12:50:00
Distinct invoices                                     25900
Known customers                                        4372
Negative quantities                                   10624
Zero quantities                                           0
Negative prices                                           2
Zero prices                                            2515
Invoice lines beginning C                              9288
Repeated rows after first occurrence                   5268
dtype: object

In [7]:
is_cancellation = (
    df["InvoiceNo"].str.upper().str.startswith("C", na=False)
)

# How cancellation markers overlap with negative quantities
display(pd.crosstab(
    df["Quantity"].lt(0),
    is_cancellation,
    rownames=["Negative quantity"],
    colnames=["Invoice begins C"]
))

# Both negative-price records
display(df.loc[df["UnitPrice"].lt(0)])

# A sample of negative quantities without a cancellation marker
display(
    df.loc[df["Quantity"].lt(0) & ~is_cancellation].head(15)
)

# A sample of zero-price records
display(df.loc[df["UnitPrice"].eq(0)].head(15))

Invoice begins C,False,True
Negative quantity,,
False,531285,0
True,1336,9288


,source_row,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299983,299985,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,<NA>,United Kingdom
299984,299986,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,<NA>,United Kingdom


,source_row,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
2406,2408,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,<NA>,United Kingdom
4347,4349,536764,84952C,NaN,-38,2010-12-02 14:42:00,0.0,<NA>,United Kingdom
7188,7190,536996,22712,NaN,-20,2010-12-03 15:30:00,0.0,<NA>,United Kingdom
7189,7191,536997,22028,NaN,-20,2010-12-03 15:30:00,0.0,<NA>,United Kingdom
7190,7192,536998,85067,NaN,-6,2010-12-03 15:30:00,0.0,<NA>,United Kingdom
7192,7194,537000,21414,NaN,-22,2010-12-03 15:32:00,0.0,<NA>,United Kingdom
7193,7195,537001,21653,NaN,-6,2010-12-03 15:33:00,0.0,<NA>,United Kingdom
7195,7197,537003,85126,NaN,-2,2010-12-03 15:33:00,0.0,<NA>,United Kingdom
7196,7198,537004,21814,NaN,-30,2010-12-03 15:34:00,0.0,<NA>,United Kingdom
7197,7199,537005,21692,NaN,-70,2010-12-03 15:35:00,0.0,<NA>,United Kingdom


,source_row,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,624,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,<NA>,United Kingdom
1970,1972,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,<NA>,United Kingdom
1971,1973,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,<NA>,United Kingdom
1972,1974,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,<NA>,United Kingdom
1987,1989,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,<NA>,United Kingdom
1988,1990,536550,85044,NaN,1,2010-12-01 14:34:00,0.0,<NA>,United Kingdom
2024,2026,536552,20950,NaN,1,2010-12-01 14:34:00,0.0,<NA>,United Kingdom
2025,2027,536553,37461,NaN,3,2010-12-01 14:35:00,0.0,<NA>,United Kingdom
2026,2028,536554,84670,NaN,23,2010-12-01 14:35:00,0.0,<NA>,United Kingdom
2406,2408,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,<NA>,United Kingdom


In [8]:
# Check ALL negative-quantity lines without a C marker
other_negative = df.loc[
    df["Quantity"].lt(0) & ~is_cancellation
]

display(pd.Series({
    "Total lines": len(other_negative),
    "Zero-price lines": other_negative["UnitPrice"].eq(0).sum(),
    "Positive-price lines": other_negative["UnitPrice"].gt(0).sum(),
    "Negative-price lines": other_negative["UnitPrice"].lt(0).sum(),
}))

# Find related bad-debt entries, including any positive adjustments
bad_debt = df["Description"].str.contains(
    "bad debt", case=False, na=False
)

display(
    df.loc[bad_debt].assign(
        line_value=lambda x: x["Quantity"] * x["UnitPrice"]
    )
)

# Inspect any non-C negative quantities with a non-zero price
display(
    other_negative.loc[
        other_negative["UnitPrice"].ne(0)
    ].head(20)
)


Total lines             1336
Zero-price lines        1336
Positive-price lines       0
Negative-price lines       0
dtype: int64

,source_row,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,line_value
299982,299984,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,<NA>,United Kingdom,11062.06
299983,299985,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,<NA>,United Kingdom,-11062.06
299984,299986,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,<NA>,United Kingdom,-11062.06


,source_row,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country


In [9]:
analysis = df.copy()

analysis["line_value"] = (
    analysis["Quantity"] * analysis["UnitPrice"]
).round(2)

analysis["is_cancellation"] = (
    analysis["InvoiceNo"]
    .str.upper()
    .str.startswith("C", na=False)
)

analysis["missing_customer"] = analysis["CustomerID"].isna()

# Flag repeated source records without deleting them
analysis["possible_duplicate"] = (
    df.drop(columns="source_row").duplicated()
)

# Leave unexpected combinations visible for review
analysis["record_type"] = "Needs review"

analysis.loc[
    analysis["Quantity"].gt(0)
    & analysis["UnitPrice"].gt(0)
    & ~analysis["is_cancellation"],
    "record_type"
] = "Positive sales / charges"

analysis.loc[
    analysis["Quantity"].lt(0)
    & analysis["UnitPrice"].gt(0)
    & analysis["is_cancellation"],
    "record_type"
] = "Cancellation / credit"

analysis.loc[
    analysis["UnitPrice"].eq(0),
    "record_type"
] = "Zero-price record"

# Apply this last so the positive bad-debt entry is also separated
analysis.loc[
    analysis["Description"].str.contains(
        "bad debt", case=False, na=False
    ),
    "record_type"
] = "Bad-debt adjustment"

record_summary = (
    analysis.groupby("record_type")
    .agg(
        lines=("source_row", "size"),
        invoices=("InvoiceNo", "nunique"),
        signed_value=("line_value", "sum"),
        missing_customer_lines=("missing_customer", "sum"),
        possible_duplicate_lines=("possible_duplicate", "sum"),
    )
)

display(record_summary.round(2))

assert record_summary["lines"].sum() == len(df)

,lines,invoices,signed_value,missing_customer_lines,possible_duplicate_lines
record_type,,,,,
Bad-debt adjustment,3,3,-11062.06,3,0
Cancellation / credit,9288,3836,-896812.49,383,37
Positive sales / charges,530103,19959,10655622.48,132219,5226
Zero-price record,2515,2155,0.00,2475,5


In [10]:
included_types = [
    "Positive sales / charges",
    "Cancellation / credit",
]

eligible = analysis["record_type"].isin(included_types)

comparison = pd.DataFrame({
    "All records retained": (
        analysis.loc[eligible]
        .groupby("record_type")["line_value"].sum()
    ),
    "Repeated rows excluded": (
        analysis.loc[
            eligible & ~analysis["possible_duplicate"]
        ]
        .groupby("record_type")["line_value"].sum()
    ),
}).reindex(included_types).fillna(0)

comparison.loc["Net recorded value"] = comparison.sum()

comparison["Difference from excluding repeats"] = (
    comparison["Repeated rows excluded"]
    - comparison["All records retained"]
)

display(comparison.round(2))

,All records retained,Repeated rows excluded,Difference from excluding repeats
record_type,,,
Positive sales / charges,10655622.48,10631048.74,-24573.74
Cancellation / credit,-896812.49,-893979.73,2832.76
Net recorded value,9758809.99,9737069.01,-21740.98


In [11]:
# A screening rule, not a reason to exclude records:
# flag codes outside the common five-digit + optional-letter format.
unusual_code = ~analysis["StockCode"].str.fullmatch(
    r"\d{5}[A-Za-z]*", na=False
)

code_review = (
    analysis.loc[eligible & unusual_code]
    .groupby("StockCode")
    .agg(
        example_descriptions=(
            "Description",
            lambda s: " | ".join(
                s.dropna().astype(str).unique()[:3]
            )
        ),
        lines=("source_row", "size"),
        signed_value=("line_value", "sum"),
    )
    .sort_values(
        "signed_value",
        key=lambda s: s.abs(),
        ascending=False,
    )
)

display(code_review.round(2))

,example_descriptions,lines,signed_value
StockCode,,,
AMAZONFEE,AMAZON FEE,34,-221520.50
DOT,DOTCOM POSTAGE,707,206245.48
M,Manual,565,-68674.19
POST,POSTAGE,1252,66230.64
CRUK,CRUK Commission,16,-7933.43
BANK CHARGES,Bank Charges,37,-7175.64
C2,CARRIAGE,143,6986.00
D,Discount,77,-5696.22
S,SAMPLES,63,-3049.39


In [12]:
# Normalise codes for classification; preserve the original StockCode.
code = analysis["StockCode"].str.strip().str.upper()

category_map = {
    "POST": "Delivery charges",
    "DOT": "Delivery charges",
    "C2": "Delivery charges",
    "AMAZONFEE": "Fees and commissions",
    "BANK CHARGES": "Fees and commissions",
    "CRUK": "Fees and commissions",
    "M": "Manual entries",
    "D": "Discounts",
    "S": "Samples",
}

analysis["item_category"] = (
    code.map(category_map).fillna("Merchandise")
)

analysis.loc[
    code.str.startswith("GIFT_", na=False),
    "item_category"
] = "Gift vouchers"

analysis.loc[
    analysis["record_type"].eq("Bad-debt adjustment"),
    "item_category"
] = "Bad-debt adjustments"

category_summary = (
    analysis.loc[eligible]
    .groupby(["item_category", "record_type"])["line_value"]
    .sum()
    .unstack(fill_value=0)
    .reindex(columns=included_types, fill_value=0)
)

category_summary["Net recorded value"] = (
    category_summary.sum(axis=1)
)

display(category_summary.round(2))

record_type,Positive sales / charges,Cancellation / credit,Net recorded value
item_category,,,
Delivery charges,291401.65,-11939.53,279462.12
Discounts,0.00,-5696.22,-5696.22
Fees and commissions,13926.09,-250555.66,-236629.57
Gift vouchers,685.81,0.00,685.81
Manual entries,78112.82,-146784.46,-68671.64
Merchandise,10271433.06,-478724.18,9792708.88
Samples,63.05,-3112.44,-3049.39


In [13]:
import sqlite3

query = (
    PROJECT_ROOT / "sql/02_category_reconciliation.sql"
).read_text()

with sqlite3.connect(PROJECT_ROOT / "data/retail.sqlite") as conn:
    sql_summary = pd.read_sql_query(
        query, conn
    ).set_index("item_category")

pd.testing.assert_frame_equal(
    sql_summary.sort_index(),
    category_summary.round(2).sort_index(),
    check_names=False,
    check_dtype=False,
    check_exact=False,
    atol=0.005,
    rtol=0,
)

print("PASS: SQL and Python category totals agree.")

PASS: SQL and Python category totals agree.


In [14]:
query = (
    PROJECT_ROOT / "sql/03_monthly_merchandise.sql"
).read_text()

with sqlite3.connect(PROJECT_ROOT / "data/retail.sqlite") as conn:
    monthly = pd.read_sql_query(query, conn)

# Monthly net values must reconcile to our merchandise total.
assert round(monthly["net_recorded_value"].sum(), 2) == round(
    category_summary.loc["Merchandise", "Net recorded value"], 2
)

display(monthly)


,month_start,is_partial_month,positive_value,credit_value,net_recorded_value,credit_to_positive_value_pct,positive_sales_invoices
0,2010-12-01,0,777931.77,17607.86,760323.91,2.26,1550
1,2011-01-01,0,671933.19,91526.30,580406.89,13.62,1081
2,2011-02-01,0,508886.20,8375.87,500510.33,1.65,1093
3,2011-03-01,0,691257.61,10675.30,680582.31,1.54,1440
4,2011-04-01,0,516218.12,33316.91,482901.21,6.45,1235
5,2011-05-01,0,741192.41,8948.23,732244.18,1.21,1668
6,2011-06-01,0,738751.87,13760.84,724991.03,1.86,1525
7,2011-07-01,0,689364.41,11409.21,677955.20,1.66,1452
8,2011-08-01,0,725513.50,22929.52,702583.98,3.16,1339
9,2011-09-01,0,1030475.36,17044.68,1013430.68,1.65,1818


In [15]:
query = (
    PROJECT_ROOT / "sql/04_credit_invoice_concentration.sql"
).read_text()

with sqlite3.connect(PROJECT_ROOT / "data/retail.sqlite") as conn:
    credit_drivers = pd.read_sql_query(query, conn)

display(credit_drivers)

,month_start,invoice_rank,InvoiceNo,invoice_credit_value,month_credit_value,share_of_month_credit_pct,credit_lines,credited_units
0,2011-01-01,1,C541433,77183.60,91526.30,84.33,1,74215
1,2011-01-01,2,C540417,1520.11,91526.30,1.66,21,669
2,2011-01-01,3,C542426,1241.43,91526.30,1.36,35,551
3,2011-04-01,1,C550456,22998.40,33316.91,69.03,5,9014
4,2011-04-01,2,C548731,1591.20,33316.91,4.78,1,624
5,2011-04-01,3,C548577,933.92,33316.91,2.80,5,632
6,2011-12-01,1,C581484,168469.60,174126.76,96.75,1,80995
7,2011-12-01,2,C580971,664.86,174126.76,0.38,3,322
8,2011-12-01,3,C580954,436.20,174126.76,0.25,13,194


In [16]:
query = (
    PROJECT_ROOT / "sql/05_large_credit_sale_candidates.sql"
).read_text()

with sqlite3.connect(PROJECT_ROOT / "data/retail.sqlite") as conn:
    sale_candidates = pd.read_sql_query(query, conn)

display(sale_candidates)

,credit_source_row,credit_invoice,StockCode,Description,credit_date,credited_units,credit_value,CustomerID,candidate_sale_source_row,candidate_sale_invoice,candidate_sale_date,minutes_between,candidate_count
0,61626,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,2011-01-18 10:17:00,74215,77183.6,12346,61621.0,541431,2011-01-18 10:01:00,16.0,1
1,160143,C550456,48185,DOORMAT FAIRY CAKE,2011-04-18 13:08:00,670,4522.5,15749,52774.0,540818,2011-01-11 12:57:00,139691.0,1
2,160144,C550456,47566B,TEA TIME PARTY BUNTING,2011-04-18 13:08:00,1300,3315.0,15749,NaN,None,None,NaN,0
3,160145,C550456,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2011-04-18 13:08:00,1930,4921.5,15749,52711.0,540815,2011-01-11 12:55:00,139693.0,1
4,160146,C550456,21175,GIN + TONIC DIET METAL SIGN,2011-04-18 13:08:00,2000,3700.0,15749,52712.0,540815,2011-01-11 12:55:00,139693.0,1
5,160147,C550456,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,2011-04-18 13:08:00,3114,6539.4,15749,52713.0,540815,2011-01-11 12:55:00,139693.0,1
6,540424,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",2011-12-09 09:27:00,80995,168469.6,16446,540423.0,581483,2011-12-09 09:15:00,12.0,1


In [17]:
query = (
    PROJECT_ROOT / "sql/06_same_day_pair_sensitivity.sql"
).read_text()

with sqlite3.connect(PROJECT_ROOT / "data/retail.sqlite") as conn:
    pair_sensitivity = pd.read_sql_query(query, conn)

display(pair_sensitivity)

,scenario,retained_lines,positive_value,credit_value,net_recorded_value,credit_to_positive_value_pct
0,Baseline: all merchandise records,536465,10271433.06,478724.18,9792708.88,4.66
1,Sensitivity: exclude two candidate same-day pairs,536461,10025779.86,233070.98,9792708.88,2.32


In [18]:
def run_query(filename):
    query = (PROJECT_ROOT / "sql" / filename).read_text()
    with sqlite3.connect(PROJECT_ROOT / "data/retail.sqlite") as conn:
        return pd.read_sql_query(query, conn)

products = run_query("07_product_summary.sql")
countries = run_query("08_country_summary.sql")

# Check complete results before displaying only the top ten.
for table in (products, countries):
    assert round(table["net_recorded_value"].sum(), 2) == 9792708.88

ratio_columns = [
    "credit_to_positive_value_pct",
    "sensitivity_credit_to_positive_value_pct",
]

display(products[[
    "StockCode",
    "example_description",
    "net_recorded_value",
] + ratio_columns].head(10))

display(countries[[
    "Country",
    "net_recorded_value",
    "net_value_share_pct",
] + ratio_columns].head(10))

,StockCode,example_description,net_recorded_value,credit_to_positive_value_pct,sensitivity_credit_to_positive_value_pct
0,22423,REGENCY CAKESTAND 3 TIER,164762.19,5.57,5.57
1,47566,PARTY BUNTING,98302.98,1.21,1.21
2,85123A,CREAM HANGING HEART T-LIGHT HOLDER,97894.50,6.34,6.34
3,85099B,JUMBO BAG RED RETROSPOT,92356.03,2.10,2.10
4,23084,RABBIT NIGHT LIGHT,66756.59,0.31,0.31
5,22086,PAPER CHAIN KIT 50'S CHRISTMAS,63791.94,1.79,1.79
6,84879,ASSORTED COLOUR BIRD ORNAMENT,58959.73,0.23,0.23
7,79321,CHILLI LIGHTS,53768.06,0.65,0.65
8,22502,PICNIC BASKET WICKER 60 PIECES,51041.37,0.75,0.75
9,22197,POPCORN HOLDER,50987.47,0.71,0.71


,Country,net_recorded_value,net_value_share_pct,credit_to_positive_value_pct,sensitivity_credit_to_positive_value_pct
0,United Kingdom,8301718.28,84.77,5.10,2.35
1,Netherlands,283479.54,2.89,0.14,0.14
2,EIRE,259663.46,2.65,4.24,4.24
3,Germany,200808.40,2.05,2.32,2.32
4,France,182163.39,1.86,1.31,1.31
5,Australia,136990.00,1.40,0.85,0.85
6,Switzerland,52505.35,0.54,1.10,1.10
7,Spain,51765.20,0.53,7.11,7.11
8,Belgium,36662.96,0.37,0.72,0.72
9,Japan,35419.79,0.36,5.34,5.34
